In [2]:
# -------------------------------
# Step 1: Import required libraries
# -------------------------------
import numpy as np          # For numerical operations like arrays, max, argmax
import gymnasium as gym     # Upgraded to Gymnasium (the maintained replacement for gym)
import time                 # To slow down the visualization rendering

# -------------------------------
# Step 2: Create the environment
# -------------------------------
# Creating the Taxi-v3 environment
# "ansi" render mode allows us to print the text-based grid map directly to the console
env = gym.make("Taxi-v4", render_mode="ansi")

print("Number of states:", env.observation_space.n)  # Total possible states (500)
print("Number of actions:", env.action_space.n)      # Total possible actions (6)

# -------------------------------
# Step 3: Global Parameters
# -------------------------------
gamma = 0.99  # Higher discount factor because Taxi journeys take more steps than FrozenLake
theta = 1e-10 # Convergence threshold for policy evaluation

# -------------------------------
# Step 4: Policy Evaluation
# -------------------------------
def policy_evaluation(policy, value_table, env, gamma, theta):
    """
    Evaluate the value function under the current policy.
    Iteratively update state values until convergence.
    """
    unwrapped_env = env.unwrapped
    while True:
        delta = 0  # Tracks maximum change in value during an iteration

        for state in range(unwrapped_env.observation_space.n):
            v = 0
            action = policy[state]  # Action chosen by current policy

            # Look at possible transitions for this (state, action)
            for prob, next_state, reward, done in unwrapped_env.P[state][action]:
                # Bellman expectation equation
                v += prob * (reward + gamma * value_table[next_state])

            delta = max(delta, abs(value_table[state] - v))
            value_table[state] = v  # Update value of state

        if delta < theta:
            break

    return value_table

# -------------------------------
# Step 5: Policy Improvement
# -------------------------------
def policy_improvement(value_table, policy, env, gamma):
    """
    Improve the policy using the current value function.
    """
    unwrapped_env = env.unwrapped
    policy_stable = True

    for state in range(unwrapped_env.observation_space.n):
        old_action = policy[state]
        action_values = np.zeros(unwrapped_env.action_space.n)  # Store values of all 6 actions

        # Try all possible actions from this state
        for action in range(unwrapped_env.action_space.n):
            for prob, next_state, reward, done in unwrapped_env.P[state][action]:
                action_values[action] += prob * (reward + gamma * value_table[next_state])

        # Choose action with highest expected value
        policy[state] = np.argmax(action_values)

        if old_action != policy[state]:
            policy_stable = False

    return policy, policy_stable

# -------------------------------
# Step 6: Policy Iteration Algorithm
# -------------------------------
def policy_iteration(env, gamma, theta):
    """
    Run policy iteration algorithm.
    Returns optimal policy and value function.
    """
    unwrapped_env = env.unwrapped
    # Start with a random initial policy for all 500 states
    policy = np.random.choice(unwrapped_env.action_space.n, size=unwrapped_env.observation_space.n)
    value_table = np.zeros(unwrapped_env.observation_space.n)

    iteration = 0
    while True:
        iteration += 1
        value_table = policy_evaluation(policy, value_table, env, gamma, theta)
        policy, policy_stable = policy_improvement(value_table, policy, env, gamma)

        if policy_stable:
            print(f"Policy converged successfully in {iteration} iterations.")
            return policy, value_table

# -------------------------------
# Step 7: Train the Agent
# -------------------------------
print("\n--- Training started via Policy Iteration ---")
optimal_policy, optimal_value = policy_iteration(env, gamma, theta)
print("--- Training complete! ---")

# -------------------------------
# Step 8: Visualize the Trained Agent
# -------------------------------
print("\n--- Testing & Visualizing the Trained Agent ---")

# Reset the environment to get a fresh random setup
state, info = env.reset()
done = False
total_reward = 0
steps = 0

# Print the starting map layout
print("Initial Map State:")
print(env.render())

# Let the agent play out its learned policy
while not done and steps < 50:  # Safety cap at 50 steps
    # Pick the best action according to our learned optimal policy table
    action = optimal_policy[state]

    # Take a step in the environment using that action
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
    steps += 1

    # Display the current state map
    print(f"Step: {steps} | Action Taken: {action} | Current Reward: {reward}")
    print(env.render())

    # Slight delay so humans can watch the taxi move frame-by-frame
    time.sleep(0.3)

print(f"Game finished in {steps} steps with a Total Reward of: {total_reward}")
env.close()


Number of states: 500
Number of actions: 6

--- Training started via Policy Iteration ---
Policy converged successfully in 16 iterations.
--- Training complete! ---

--- Testing & Visualizing the Trained Agent ---
Initial Map State:
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+


Step: 1 | Action Taken: 3 | Current Reward: -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (West)

Step: 2 | Action Taken: 1 | Current Reward: -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)

Step: 3 | Action Taken: 1 | Current Reward: -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (North)

Step: 4 | Action Taken: 3 | Current Reward: -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (West)

Step: 5 | Action Taken: 4 | Current Reward: -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B